# STEP2. 변수 생성 (Variable Generation)

## 분석 질문
- STEP1에서 병합한 원본 레벨 패널로부터 산업별 수익률·KOSPI 수익률·금리 변화분·장단기 금리차 등 회귀분석에 필요한 파생변수를 생성할 수 있는가?
- 관련 가설: 해당 없음 (H1·H2 공통 인프라)

## 입력
- 입력 파일: `data/processed/industry_rate_panel_raw.csv` (STEP1에서 저장한 원본 레벨 병합 패널)
- 이전 STEP 결과: STEP1(`01_data_preparation.ipynb`), `data/interim/kospi_daily.csv`, `data/interim/yields_daily.csv`

## 출력
- Table: `data/processed/industry_rate_panel_daily.csv`
- Summary: 없음

## 실행 환경
- VS Code(로컬) 환경에서 실행한다.

## 다음 단계
- `03_eda_industry.ipynb`에서 이 노트북이 저장한 `industry_rate_panel_daily.csv`로 기초통계·시계열·상관관계·데이터 검증을 수행한다.

## 참고
- 파생변수 생성 함수는 `src/data_prep.py`의 `add_return_and_rate_change_vars`에 정리되어 있다.
- KOSPI 수익률·금리 변화분은 반드시 KOSPI·금리 고유의 날짜 시퀀스에서 계산한 뒤 병합해야 기존 노트북과 동일한 값이 나온다(자세한 이유는 `src/data_prep.py` docstring 참고).

## 준비: 실행 환경 설정
### 목적
- 저장소 루트를 자동으로 찾아 `config`, `src.data_prep`를 불러온다.

In [ ]:
import pandas as pd
# 저장소 루트를 찾아 config를 불러온다 (실행 위치/편집기와 무관하게 동작).
# 저장소 루트에서 `pip install -e .` 를 한 번 실행했다면 아래 탐색은 건너뛴다.
import sys
from pathlib import Path

_anchors = []
if "__vsc_ipynb_file__" in globals():          # VS Code Notebook
    _anchors.append(Path(globals()["__vsc_ipynb_file__"]).resolve().parent)
if globals().get("_dh"):                        # IPython 커널 시작 폴더 (Jupyter/Lab/Colab)
    _anchors.append(Path(globals()["_dh"][0]).resolve())
_anchors.append(Path.cwd().resolve())           # 최후 수단

# config.py 와 requirements.txt 를 '둘 다' 가진 폴더만 저장소 루트로 인정한다.
# 마커를 하나만 보면 다른 폴더의 동명 config.py 를 잘못 집을 수 있다.
_root = next(
    (p for a in _anchors for p in [a, *a.parents]
     if (p / "config.py").is_file() and (p / "requirements.txt").is_file()),
    None,
)
# 이미 sys.path에 있더라도 맨 앞으로 올린다. cwd(예: 바탕화면)가 앞에 있으면
# 같은 이름의 다른 config.py 가 먼저 잡히기 때문이다.
if _root is not None:
    if str(_root) in sys.path:
        sys.path.remove(str(_root))
    sys.path.insert(0, str(_root))

import config

if _root is not None and Path(config.__file__).resolve().parent != _root:
    raise ImportError(
        f"의도하지 않은 config.py 가 import 되었습니다: {config.__file__}\n"
        f"저장소 루트({_root})에서 `pip install -e .` 를 실행하세요."
    )

from src import data_prep as dp

# 개인 PC 절대경로를 출력에 남기지 않는다 — 저장소 폴더명만 확인한다.
print("저장소 루트:", config.PROJECT_ROOT.name)

## 1. 이전 STEP 데이터 불러오기
### 목적
- `01_data_preparation.ipynb`(STEP1)에서 저장한 원본 레벨 패널과 중간 데이터(KOSPI, 국고채)를 불러온다.

In [2]:
raw_panel = pd.read_csv(config.PROCESSED_DIR / "industry_rate_panel_raw.csv", encoding="utf-8-sig")
raw_panel["date"] = pd.to_datetime(raw_panel["date"])

kospi_daily = pd.read_csv(config.INTERIM_DIR / "kospi_daily.csv", encoding="utf-8-sig")
kospi_daily["date"] = pd.to_datetime(kospi_daily["date"])

yields_daily = pd.read_csv(config.INTERIM_DIR / "yields_daily.csv", encoding="utf-8-sig")
yields_daily["date"] = pd.to_datetime(yields_daily["date"])

print("raw_panel   :", raw_panel.shape)
print("kospi_daily :", kospi_daily.shape)
print("yields_daily:", yields_daily.shape)
raw_panel.head()

raw_panel   : (17164, 7)
kospi_daily : (2452, 2)
yields_daily: (2462, 3)


,date,industry,industry_index,kospi_index,y3,y10,source_file
0,2016-01-04,건설업,104.43,1918.76,1.634,2.039,16-17.csv
1,2016-01-04,금융업,408.22,1918.76,1.634,2.039,16-17.csv
2,2016-01-04,운수장비,1731.95,1918.76,1.634,2.039,16-17.csv
3,2016-01-04,의약품,8411.35,1918.76,1.634,2.039,16-17.csv
4,2016-01-04,전기전자,9239.12,1918.76,1.634,2.039,16.1~17.12.csv


## 2. 파생변수 생성 (수익률·금리 변화분·금리차)
### 목적
- 산업별 일별 수익률(`industry_return`), KOSPI 일별 수익률(`kospi_return`), 국고채 3년 금리 변화분(`d_y3`), 국고채 10년 금리 변화분(`d_y10`), 장단기 금리차(`term_spread` = y10 - y3), 장단기 금리차 변화분(`d_term_spread`)을 한 번에 생성한다
  (`add_return_and_rate_change_vars` 내부에서 각각 `pct_change()`/`diff()`로 계산).

In [3]:
final_panel = dp.add_return_and_rate_change_vars(raw_panel, kospi_daily, yields_daily)
print("final_panel:", final_panel.shape)
final_panel.head()

final_panel: (17164, 12)


,date,industry,industry_index,industry_return,kospi_index,kospi_return,y3,d_y3,y10,d_y10,term_spread,d_term_spread
0,2016-01-04,건설업,104.43,NaN,1918.76,NaN,1.634,NaN,2.039,NaN,0.405,NaN
1,2016-01-04,금융업,408.22,NaN,1918.76,NaN,1.634,NaN,2.039,NaN,0.405,NaN
2,2016-01-04,운수장비,1731.95,NaN,1918.76,NaN,1.634,NaN,2.039,NaN,0.405,NaN
3,2016-01-04,의약품,8411.35,NaN,1918.76,NaN,1.634,NaN,2.039,NaN,0.405,NaN
4,2016-01-04,전기전자,9239.12,NaN,1918.76,NaN,1.634,NaN,2.039,NaN,0.405,NaN


## 3. 파생변수 검증
### 목적
- 생성된 파생변수의 결측치 패턴(각 시계열 첫 거래일 1행씩 예상)과, 장단기 금리차 항등식(`term_spread = y10 - y3`, `d_term_spread = d_y10 - d_y3`)이 성립하는지 확인한다.

In [4]:
VARS = ["industry_return", "kospi_return", "d_y3", "d_y10", "term_spread", "d_term_spread"]
print("결측치(각 시계열 첫 거래일 1행씩 예상):")
print(final_panel[VARS].isna().sum())
print()
print("장단기 금리차 항등식 확인 (term_spread = y10 - y3):",
      (final_panel["term_spread"] - (final_panel["y10"] - final_panel["y3"])).abs().max())
print("장단기 금리차 변화분 항등식 확인 (d_term_spread = d_y10 - d_y3):",
      (final_panel["d_term_spread"] - (final_panel["d_y10"] - final_panel["d_y3"])).abs().max())

결측치(각 시계열 첫 거래일 1행씩 예상):
industry_return    7
kospi_return       7
d_y3               7
d_y10              7
term_spread        0
d_term_spread      7
dtype: int64

장단기 금리차 항등식 확인 (term_spread = y10 - y3): 0.0
장단기 금리차 변화분 항등식 확인 (d_term_spread = d_y10 - d_y3): 0.0


## 4. 최종 변수 생성 데이터 저장
### 목적
- 파생변수가 포함된 최종 패널(`industry_rate_panel_daily.csv`)을 저장한다.

In [ ]:
final_path = dp.save_final_panel(final_panel)
# 개인 PC 절대경로를 출력에 남기지 않는다 (config._rel: 저장소 루트 기준 상대경로).
print("최종 변수 생성 데이터 저장 완료:", config._rel(final_path))

## STEP2 요약

### Takeaway
- 산업별 일별 수익률, KOSPI 일별 수익률, 국고채 3년/10년 금리 변화분을 생성했다.
- 장단기 금리차(`term_spread`)와 그 변화분(`d_term_spread`)을 생성했다.
- 최종 변수 생성 데이터(`industry_rate_panel_daily.csv`)를 저장했다.
- 다음 단계: `03_eda_industry.ipynb`에서 기초통계 · 시계열 · 상관관계 · 데이터 검증을 수행한다.